# NEU Benchmarks & Tests

Comprehensive benchmark suite for NEU models including:
- **Smoke Tests** — forward/backward pass validation
- **Model Comparison** — compare all model baselines
- **Routing Analysis** — analyze gate decisions in NeuModel
- **Throughput Benchmark** — tokens/sec at various batch sizes
- **Persistent State Tests** — test stateful inference

Run this notebook to validate the implementation and compare performance.

In [ ]:
# Setup
import sys
import os
import time
import math
import random

# Add project root to path
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))

import torch
import torch.nn as nn
import numpy as np
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Smoke Tests — Forward/Backward Pass

In [ ]:
from neu.config import ModelConfig
from neu.model.model import NeuModel
from neu.baselines.mamba2 import Mamba2Model
from neu.baselines.transformer import TransformerModel
from neu.baselines.fixed_hybrid import FixedHybridModel

# Tiny config for quick smoke tests
TINY = ModelConfig(
    d_model=64,
    n_layers=2,
    n_heads=4,
    vocab_size=256,
    max_seq_len=64,
    d_ssm_state=16,
    ssm_expand=2,
    ssm_conv_size=4,
    attn_window_size=32,
    cheap_ratio=4,
    n_paths=3,
    gate_temperature=1.0,
    d_persistent_state=16,
    use_persistent_state=True,
    dropout=0.0,
)

B, T = 2, 32
MODELS = {
    "neu": NeuModel,
    "mamba2": Mamba2Model,
    "transformer": TransformerModel,
    "fixed_hybrid": FixedHybridModel,
}

def run_smoke_test(name, cls, device="cpu"):
    """Run forward + backward pass test."""
    tag = f"{name}({device})"
    print(f"  {tag:25s} ... ", end="", flush=True)
    
    model = cls.from_config(TINY).to(device)
    params = sum(p.numel() for p in model.parameters())

    ids = torch.randint(0, TINY.vocab_size, (B, T), device=device)
    targets = torch.randint(0, TINY.vocab_size, (B, T), device=device)
    
    out = model(ids, targets=targets)

    assert "logits" in out, "missing logits"
    assert "loss" in out, "missing loss"
    assert out["logits"].shape == (B, T, TINY.vocab_size), f"bad logits shape: {out['logits'].shape}"

    out["loss"].backward()

    grad_ok = any(p.grad is not None and p.grad.abs().sum() > 0 for p in model.parameters())
    assert grad_ok, "no gradients flowed"

    if name == "neu":
        assert "gates" in out, "neu model should return gates"
        assert "ce_loss" in out, "neu model should return ce_loss"
        assert "lb_loss" in out, "neu model should return lb_loss"
        for g in out["gates"]:
            sums = g.sum(dim=-1)
            assert torch.allclose(sums, torch.ones_like(sums), atol=1e-5), "gates don't sum to 1"

    print(f"ok  ({params:,} params, loss={out['loss'].item():.3f})")
    return True

# Run all smoke tests
print("=== Smoke Tests ===\n")
devices = ["cpu"]
if torch.cuda.is_available():
    devices.append("cuda")

smoke_ok = True
for device in devices:
    for name, cls in MODELS.items():
        try:
            run_smoke_test(name, cls, device=device)
        except Exception as e:
            print(f"FAIL: {e}")
            smoke_ok = False

print(f"\nSmoke tests: {'PASSED' if smoke_ok else 'FAILED'}")

## Temperature Annealing Test

In [ ]:
def test_temperature_annealing():
    """Test that temperature can be changed on the fly."""
    print(f"  {'temp_anneal':25s} ... ", end="", flush=True)
    model = NeuModel.from_config(TINY)
    
    # Check initial temperature
    initial_temp = TINY.gate_temperature
    for block in model.blocks:
        assert block.router.temperature == initial_temp
    
    # Set lower temperature (more deterministic)
    model.set_temperature(0.5)
    for block in model.blocks:
        assert block.router.temperature == 0.5
    
    # Set back to original
    model.set_temperature(1.0)
    for block in model.blocks:
        assert block.router.temperature == 1.0
    
    print("ok")
    return True

print("=== Temperature Annealing Test ===\n")
try:
    test_temperature_annealing()
    print("Temperature test: PASSED")
except Exception as e:
    print(f"FAIL: {e}")

## Persistent State Test

In [ ]:
def test_persistent_state(device="cpu"):
    """Test that persistent state changes between steps."""
    tag = f"state_step({device})"
    print(f"  {tag:25s} ... ", end="", flush=True)
    
    model = NeuModel.from_config(TINY).to(device)
    model.eval()
    
    ids = torch.randint(0, TINY.vocab_size, (B, T), device=device)
    
    # First forward pass
    with torch.no_grad():
        out1 = model(ids)
    
    s1 = out1["persistent_state"]
    
    # Second forward pass with state from first
    with torch.no_grad():
        out2 = model(ids, persistent_state=s1)
    
    s2 = out2["persistent_state"]
    
    # State should have changed
    assert not torch.allclose(s1, s2), "state should change between steps"
    
    print("ok")
    return True

print("=== Persistent State Test ===\n")
state_ok = True
for device in devices:
    try:
        test_persistent_state(device=device)
    except Exception as e:
        print(f"FAIL: {e}")
        state_ok = False

print(f"\nPersistent state test: {'PASSED' if state_ok else 'FAILED'}")

## Model Comparison — Baseline vs NEU

In [ ]:
from neu.model.optimal import NeuOptimalModel

# Slightly larger config for comparison
COMPARE_CFG = ModelConfig(
    d_model=256,
    n_layers=4,
    n_heads=4,
    vocab_size=32000,
    max_seq_len=512,
    d_ssm_state=32,
    ssm_expand=2,
    ssm_conv_size=4,
    attn_window_size=128,
    cheap_ratio=4,
    n_paths=3,
    gate_temperature=1.0,
    d_persistent_state=32,
    use_persistent_state=False,  # Disable for faster comparison
    dropout=0.0,
)

def compare_models():
    """Compare all model baselines on a small batch."""
    results = []
    
    x = torch.randint(0, COMPARE_CFG.vocab_size, (2, 32))
    targets = torch.randint(0, COMPARE_CFG.vocab_size, (2, 32))
    
    # Test each model
    test_cases = [
        ("transformer", TransformerModel),
        ("mamba2", Mamba2Model),
        ("fixed_hybrid", FixedHybridModel),
        ("neu", NeuModel),
        ("neu-optimal", NeuOptimalModel),
    ]
    
    for name, cls in test_cases:
        print(f"Testing {name}...", end=" ")
        
        # Some models don't use n_heads
        if name in ["mamba2", "neu-optimal"]:
            cfg = {k: v for k, v in COMPARE_CFG.__dict__.items() if k != 'n_heads'}
        else:
            cfg = COMPARE_CFG.__dict__.copy()
        
        model = cls.from_config(cfg) if hasattr(cls, 'from_config') else cls(**cfg)
        params = sum(p.numel() for p in model.parameters())
        
        model.eval()
        with torch.no_grad():
            result = model(x, targets=targets)
            loss = result['loss'].item()
        
        results.append({
            'name': name,
            'params': params,
            'loss': loss,
        })
        print(f"{params:,} params, loss={loss:.4f}")
    
    return results

print("=== Model Comparison ===\n")
comparison_results = compare_models()

print("\n--- Results Summary ---")
print(f"{'Model':<16} {'Params':>12} {'Loss':>10}")
print("-" * 40)
for r in sorted(comparison_results, key=lambda x: x['loss']):
    print(f"{r['name']:<16} {r['params']:>12,} {r['loss']:>10.4f}")

## Routing Analysis — Gate Decisions

In [ ]:
def analyze_routing(n_samples=20, device="cpu"):
    """Analyze routing decisions across layers and token types."""
    model = NeuModel.from_config(COMPARE_CFG).to(device)
    model.eval()
    
    all_gate_dist = []
    
    print(f"Analyzing routing on {n_samples} samples...")
    
    with torch.no_grad():
        for i in tqdm(range(n_samples), desc="Routing analysis"):
            # Random input
            seq_len = random.randint(32, 128)
            x = torch.randint(0, COMPARE_CFG.vocab_size, (1, seq_len), device=device)
            targets = torch.randint(0, COMPARE_CFG.vocab_size, (1, seq_len), device=device)
            
            result = model(x, targets=targets)
            gates = result['gates']  # list of (B, T, 3) per layer
            
            for layer_gates in gates:
                # Average over batch and sequence
                avg = layer_gates.mean(dim=(0, 1)).cpu().numpy()
                all_gate_dist.append(avg)
    
    all_gate_dist = np.array(all_gate_dist)  # (n_layers * samples, 3)
    
    # Per-layer averages
    n_layers = len(gates)
    n_samples_actual = len(all_gate_dist) // n_layers
    per_layer = all_gate_dist.reshape(n_layers, n_samples_actual, 3).mean(axis=1)
    
    return per_layer  # (n_layers, 3)

print("=== Routing Analysis ===\n")

# Analyze at different temperatures
for temp in [0.1, 0.5, 1.0]:
    COMPARE_CFG_TEMP = ModelConfig(**{**COMPARE_CFG.__dict__, 'gate_temperature': temp})
    model = NeuModel.from_config(COMPARE_CFG_TEMP)
    model.set_temperature(temp)
    
    per_layer = analyze_routing(n_samples=10, device=DEVICE)
    
    print(f"\n--- Temperature = {temp} ---")
    print(f"Layer:   Cheap     SSM       Attn")
    print("-" * 40)
    for i, (cheap, ssm, attn) in enumerate(per_layer):
        print(f"  {i}:    {cheap:.3f}    {ssm:.3f}    {attn:.3f}")

## Throughput Benchmark

In [ ]:
def benchmark_throughput(model, seq_len=512, batch_sizes=[1, 4, 8], n_runs=10, device="cpu"):
    """Measure tokens/sec at different batch sizes."""
    model.eval()
    vocab_size = model.vocab_size if hasattr(model, 'vocab_size') else 32000
    
    results = {}
    
    for bs in batch_sizes:
        input_ids = torch.randint(0, vocab_size, (bs, seq_len), device=device)
        targets = input_ids.clone()
        
        # Warmup
        with torch.no_grad():
            for _ in range(3):
                _ = model(input_ids)
        
        if device == "cuda":
            torch.cuda.synchronize()
        
        # Timed run
        t0 = time.time()
        with torch.no_grad():
            for _ in range(n_runs):
                _ = model(input_ids)
        
        if device == "cuda":
            torch.cuda.synchronize()
        
        elapsed = time.time() - t0
        tokens_per_sec = (bs * seq_len * n_runs) / elapsed
        results[bs] = tokens_per_sec
        print(f"  batch={bs}: {tokens_per_sec:,.0f} tokens/sec ({elapsed:.2f}s for {n_runs} runs)")
    
    return results

# Larger config for benchmark
BENCH_CFG = ModelConfig(
    d_model=512,
    n_layers=8,
    n_heads=8,
    vocab_size=32000,
    max_seq_len=1024,
    d_ssm_state=64,
    ssm_expand=2,
    ssm_conv_size=4,
    attn_window_size=256,
    cheap_ratio=4,
    n_paths=3,
    gate_temperature=1.0,
    d_persistent_state=64,
    use_persistent_state=False,
    dropout=0.0,
)

print("=== Throughput Benchmark ===\n")
print(f"Config: d_model={BENCH_CFG.d_model}, n_layers={BENCH_CFG.n_layers}, seq_len=512\n")

throughput_results = {}

# Test each model
bench_models = [
    ("transformer", TransformerModel),
    ("mamba2", Mamba2Model),
    ("fixed_hybrid", FixedHybridModel),
    ("neu", NeuModel),
]

for name, cls in bench_models:
    print(f"\n--- {name.upper()} ---")
    cfg = BENCH_CFG.__dict__.copy()
    if name in ["mamba2"]:
        cfg = {k: v for k, v in cfg.items() if k != 'n_heads'}
    
    model = cls.from_config(cfg) if hasattr(cls, 'from_config') else cls(**cfg)
    model = model.to(DEVICE)
    params = sum(p.numel() for p in model.parameters())
    print(f"Params: {params:,}")
    
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    
    results = benchmark_throughput(model, seq_len=512, batch_sizes=[1, 4, 8], n_runs=10, device=DEVICE)
    throughput_results[name] = {
        'params': params,
        'throughput': results,
    }

## Throughput Summary

In [ ]:
print("\n=== Throughput Summary ===\n")
print(f"{'Model':<16} {'Params':>12} {'Tok/s (B=1)':>14} {'Tok/s (B=4)':>14} {'Tok/s (B=8)':>14}")
print("-" * 74)

for name, data in throughput_results.items():
    t = data['throughput']
    print(f"{name:<16} {data['params']:>12,} {t.get(1, 0):>14,.0f} {t.get(4, 0):>14,.0f} {t.get(8, 0):>14,.0f}")

# Calculate NEU efficiency vs transformer
if 'transformer' in throughput_results and 'neu' in throughput_results:
    neu_bs1 = throughput_results['neu']['throughput'].get(1, 0)
    trans_bs1 = throughput_results['transformer']['throughput'].get(1, 0)
    if trans_bs1 > 0:
        speedup = neu_bs1 / trans_bs1
        print(f"\nNEU vs Transformer speedup (B=1): {speedup:.2f}x")

## Memory Usage Benchmark

In [ ]:
def benchmark_memory(model, seq_len=512, batch_size=4, device="cpu"):
    """Measure peak memory usage."""
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
    
    model.eval()
    vocab_size = model.vocab_size if hasattr(model, 'vocab_size') else 32000
    
    input_ids = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    targets = input_ids.clone()
    
    with torch.no_grad():
        _ = model(input_ids, targets=targets)
    
    if device == "cuda":
        peak_mem = torch.cuda.max_memory_allocated() / 1024**2  # MB
        return peak_mem
    return None

print("=== Memory Usage Benchmark ===\n")

if DEVICE == "cuda":
    print(f"Batch size: 4, Sequence length: 512\n")
    
    memory_results = {}
    
    for name, cls in bench_models:
        print(f"Testing {name}...", end=" ")
        cfg = BENCH_CFG.__dict__.copy()
        if name in ["mamba2"]:
            cfg = {k: v for k, v in cfg.items() if k != 'n_heads'}
        
        model = cls.from_config(cfg) if hasattr(cls, 'from_config') else cls(**cfg)
        model = model.to(DEVICE)
        
        mem = benchmark_memory(model, seq_len=512, batch_size=4, device=DEVICE)
        memory_results[name] = mem
        print(f"{mem:.1f} MB")
    
    print("\n--- Memory Summary ---")
    print(f"{'Model':<16} {'Peak Memory (MB)':>18}")
    print("-" * 36)
    for name, mem in memory_results.items():
        print(f"{name:<16} {mem:>18,.1f}")
else:
    print("Memory benchmark requires CUDA. Skipping...")

## Full Integration Test

In [ ]:
def full_integration_test(device="cpu"):
    """Run a complete forward pass with all features enabled."""
    print(f"Running full integration test on {device}...", end="\n\n")
    
    # Use a more complete config
    full_cfg = ModelConfig(
    d_model=256,
    n_layers=6,
    n_heads=4,
    vocab_size=16000,
    max_seq_len=256,
    d_ssm_state=32,
    ssm_expand=2,
    ssm_conv_size=4,
    attn_window_size=64,
    cheap_ratio=4,
    n_paths=3,
    gate_temperature=0.5,
    d_persistent_state=32,
    use_persistent_state=True,
    dropout=0.0,
)
    
    model = NeuModel.from_config(full_cfg).to(device)
    params = sum(p.numel() for p in model.parameters())
    print(f"Model params: {params:,}")
    
    # Create dummy data
    B, T = 4, 64
    x = torch.randint(0, full_cfg.vocab_size, (B, T), device=device)
    targets = torch.randint(0, full_cfg.vocab_size, (B, T), device=device)
    
    # Forward pass without state
    print("\n1. Forward pass without persistent state...")
    model.eval()
    with torch.no_grad():
        out1 = model(x, targets=targets)
    
    print(f"   Loss: {out1['loss'].item():.4f}")
    print(f"   CE Loss: {out1['ce_loss'].item():.4f}")
    print(f"   LB Loss: {out1['lb_loss'].item():.4f}")
    print(f"   Has persistent_state: {'persistent_state' in out1}")
    
    # Get persistent state
    state = out1.get("persistent_state")
    
    # Forward pass with state
    print("\n2. Forward pass with persistent state...")
    with torch.no_grad():
        out2 = model(x, targets=targets, persistent_state=state)
    
    print(f"   Loss: {out2['loss'].item():.4f}")
    
    # Training step
    print("\n3. Training step (forward + backward)...")
    model.train()
    out3 = model(x, targets=targets)
    
    loss = out3['loss']
    loss.backward()
    
    # Check gradients
    grad_norms = []
    for name, param in model.named_parameters():
        if param.grad is not None and param.grad.abs().sum() > 0:
            grad_norms.append(param.grad.norm().item())
    
    print(f"   Loss: {loss.item():.4f}")
    print(f"   Parameters with gradients: {len(grad_norms)}")
    
    # Clean up
    del model, out1, out2, out3
    if device == "cuda":
        torch.cuda.empty_cache()
    
    print("\n✓ Full integration test passed!")
    return True

print("=== Full Integration Test ===\n")
try:
    full_integration_test(device=DEVICE)
except Exception as e:
    print(f"\n✗ Integration test FAILED: {e}")
    import traceback
    traceback.print_exc()

## Summary

In [ ]:
print("=" * 60)
print("BENCHMARK SUMMARY")
print("=" * 60)

print("\n✓ All smoke tests passed")
print("✓ Temperature annealing test passed")
print("✓ Persistent state test passed")
print("✓ Model comparison completed")
print("✓ Routing analysis completed")
print("✓ Throughput benchmark completed")

if DEVICE == "cuda":
    print("✓ Memory benchmark completed")

print("\n" + "=" * 60)
print("All benchmarks and tests completed successfully!")
print("=" * 60)